#JoshaLynn Worth
#04.13.2026

In [160]:
#imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from datetime import datetime

In [161]:
#read in the data set
df = pd.read_csv("train.csv")


#Part 0

In [162]:
#Exploring the data
print(df.head())

   Unnamed: 0                              Name    Location  Year  \
0           1  Hyundai Creta 1.6 CRDi SX Option        Pune  2015   
1           2                      Honda Jazz V     Chennai  2011   
2           3                 Maruti Ertiga VDI     Chennai  2012   
3           4   Audi A4 New 2.0 TDI Multitronic  Coimbatore  2013   
4           6            Nissan Micra Diesel XV      Jaipur  2013   

   Kilometers_Driven Fuel_Type Transmission Owner_Type     Mileage   Engine  \
0              41000    Diesel       Manual      First  19.67 kmpl  1582 CC   
1              46000    Petrol       Manual      First    13 km/kg  1199 CC   
2              87000    Diesel       Manual      First  20.77 kmpl  1248 CC   
3              40670    Diesel    Automatic     Second   15.2 kmpl  1968 CC   
4              86999    Diesel       Manual      First  23.08 kmpl  1461 CC   

       Power  Seats  New_Price  Price  
0  126.2 bhp    5.0        NaN  12.50  
1   88.7 bhp    5.0  8.61 Lakh

In [163]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5847 entries, 0 to 5846
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         5847 non-null   int64  
 1   Name               5847 non-null   object 
 2   Location           5847 non-null   object 
 3   Year               5847 non-null   int64  
 4   Kilometers_Driven  5847 non-null   int64  
 5   Fuel_Type          5847 non-null   object 
 6   Transmission       5847 non-null   object 
 7   Owner_Type         5847 non-null   object 
 8   Mileage            5845 non-null   object 
 9   Engine             5811 non-null   object 
 10  Power              5811 non-null   object 
 11  Seats              5809 non-null   float64
 12  New_Price          815 non-null    object 
 13  Price              5847 non-null   float64
dtypes: float64(2), int64(3), object(9)
memory usage: 639.6+ KB
None


In [164]:
print(df.columns)

Index(['Unnamed: 0', 'Name', 'Location', 'Year', 'Kilometers_Driven',
       'Fuel_Type', 'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power',
       'Seats', 'New_Price', 'Price'],
      dtype='object')


#Part A & B

In [165]:
#how many null values
print(df.isnull().sum())

Unnamed: 0              0
Name                    0
Location                0
Year                    0
Kilometers_Driven       0
Fuel_Type               0
Transmission            0
Owner_Type              0
Mileage                 2
Engine                 36
Power                  36
Seats                  38
New_Price            5032
Price                   0
dtype: int64


In [166]:
#percentage of values are null
print((df.isnull().sum() / len(df)) * 100)

Unnamed: 0            0.000000
Name                  0.000000
Location              0.000000
Year                  0.000000
Kilometers_Driven     0.000000
Fuel_Type             0.000000
Transmission          0.000000
Owner_Type            0.000000
Mileage               0.034206
Engine                0.615700
Power                 0.615700
Seats                 0.649906
New_Price            86.061228
Price                 0.000000
dtype: float64


In [167]:
#viewing some of the null data since their is such small amounts of the datasets missing i thib
#df[df["Mileage"].isnull()]
#df[df["Engine"].isnull()]
#df[df["Power"].isnull()]
#df[df["Seats"].isnull()]

In [168]:
# Columns that need unit cleaning
cols_with_units = ["Mileage", "Engine", "Power"]

for col in cols_with_units:
    df[col] = df[col].astype(str).str.extract(r'(\d+\.?\d*)')
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Columns to fill with median
cols_to_fill = ["Mileage", "Engine", "Power", "Seats"]

for col in cols_to_fill:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

/tmp/ipykernel_12311/3277740896.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_val, inplace=True)


Some of the columns, such as Mileage, Engine, and Power, had values with units like kmpl, CC, and bhp. I removed these units by extracting just the numeric part of the values and converting them into numeric data types. This step was important because I needed the data to be in numeric form before I could calculate things like the median and use it for further analysis.

In [169]:
# Check results
print("\nRemaining missing values:")
print(df[cols_to_fill].isnull().sum())


Remaining missing values:
Mileage    0
Engine     0
Power      0
Seats      0
dtype: int64


I think this was justified becuse using mean would be rsiky as it could esaily be skewed but the amound of data missing for these were small enought simply replcing with median should be fine.

In [170]:
#Tring to see if the valus were misread but seems to be all NaN
#print(df[df["New_Price"].isnull()].head(20))
df.drop("New_Price", axis=1, inplace=True)
#make sure it worked
#print(df.columns)


Decided to drop New_Price as such a large portion of the data was missing and it would have been skewed/biased if I had tried to impute such a large amount of missing data.

#Part C

In [171]:
# Initialize encoder
encoder = OneHotEncoder(drop="first", sparse_output=False)

# Fit and transform
encoded_data = encoder.fit_transform(df[["Fuel_Type", "Transmission"]])

# Convert to DataFrame with proper column names
encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(["Fuel_Type", "Transmission"])
)

# Reset index to align properly
encoded_df.index = df.index

# Drop original columns and join new ones
df = df.drop(["Fuel_Type", "Transmission"], axis=1)
df = pd.concat([df, encoded_df], axis=1)

In [172]:
# Verifying it worked
print(df.head())
print(df.columns)

   Unnamed: 0                              Name    Location  Year  \
0           1  Hyundai Creta 1.6 CRDi SX Option        Pune  2015   
1           2                      Honda Jazz V     Chennai  2011   
2           3                 Maruti Ertiga VDI     Chennai  2012   
3           4   Audi A4 New 2.0 TDI Multitronic  Coimbatore  2013   
4           6            Nissan Micra Diesel XV      Jaipur  2013   

   Kilometers_Driven Owner_Type  Mileage  Engine   Power  Seats  Price  \
0              41000      First    19.67  1582.0  126.20    5.0  12.50   
1              46000      First    13.00  1199.0   88.70    5.0   4.50   
2              87000      First    20.77  1248.0   88.76    7.0   6.00   
3              40670     Second    15.20  1968.0  140.80    5.0  17.74   
4              86999      First    23.08  1461.0   63.10    5.0   3.50   

   Fuel_Type_Electric  Fuel_Type_Petrol  Transmission_Manual  
0                 0.0               0.0                  1.0  
1             

#Part D

At first I was confused because I wanna enginner a feature on price but it seems that the new price is in lakh? !

In [173]:
current_year = datetime.now().year
#using the example enginered feture I want to see a price_per_age
df["Age"] = current_year - df["Year"]

In [174]:
#calc price_per_Age
df["Price_per_Age"] = df["Price"] / df["Age"]
#incase anycars are super new
df["Price_per_Age"] = df["Price"] / df["Age"].replace(0, 1)
#view some
print(df[["Price", "Age", "Price_per_Age"]].head())

   Price  Age  Price_per_Age
0  12.50   11       1.136364
1   4.50   15       0.300000
2   6.00   14       0.428571
3  17.74   13       1.364615
4   3.50   13       0.269231


#Part E

Showing an example of each in the same order

In [175]:
#Select
selected_df = df[["Name", "Mileage"]]
print(selected_df.head())

                               Name  Mileage
0  Hyundai Creta 1.6 CRDi SX Option    19.67
1                      Honda Jazz V    13.00
2                 Maruti Ertiga VDI    20.77
3   Audi A4 New 2.0 TDI Multitronic    15.20
4            Nissan Micra Diesel XV    23.08


In [176]:
#Filter
filtered_df = df[(df["Year"] > 2018) & (df["Price"] > 15)]
print(filtered_df.head())

     Unnamed: 0                                         Name    Location  \
65           67     Mercedes-Benz C-Class Progressive C 220d  Coimbatore   
226         235               Toyota Innova Crysta 2.8 GX AT       Kochi   
343         357                  Skoda Superb L&K 1.8 TSI AT       Kochi   
524         544  Mercedes-Benz New C-Class Progressive C 200       Kochi   
578         599               Toyota Innova Crysta 2.8 ZX AT  Coimbatore   

     Year  Kilometers_Driven Owner_Type  Mileage  Engine   Power  Seats  \
65   2019              15369      First     0.00  1950.0  194.00    5.0   
226  2019              14165      First    11.36  2755.0  171.50    7.0   
343  2019              13747      First    14.67  1798.0  177.46    5.0   
524  2019              13190      First     0.00  1950.0  181.43    5.0   
578  2019              40674      First    11.36  2755.0  171.50    7.0   

     Price  Fuel_Type_Electric  Fuel_Type_Petrol  Transmission_Manual  Age  \
65   35.67    

In [177]:
#Rename
renamed_df = df.rename(columns={
    "Kilometers_Driven": "KM_Driven"
})
print(renamed_df.head(2))

   Unnamed: 0                              Name Location  Year  KM_Driven  \
0           1  Hyundai Creta 1.6 CRDi SX Option     Pune  2015      41000   
1           2                      Honda Jazz V  Chennai  2011      46000   

  Owner_Type  Mileage  Engine  Power  Seats  Price  Fuel_Type_Electric  \
0      First    19.67  1582.0  126.2    5.0   12.5                 0.0   
1      First    13.00  1199.0   88.7    5.0    4.5                 0.0   

   Fuel_Type_Petrol  Transmission_Manual  Age  Price_per_Age  
0               0.0                  1.0   11       1.136364  
1               1.0                  1.0   15       0.300000  


In [180]:
#Mutate
df["KM_per_Year"] = df["Kilometers_Driven"] / (df["Age"] + 1)
print(df[["Kilometers_Driven", "Age", "KM_per_Year"]].head())


   Kilometers_Driven  Age  KM_per_Year
0              41000   11  3416.666667
1              46000   15  2875.000000
2              87000   14  5800.000000
3              40670   13  2905.000000
4              86999   13  6214.214286


In [182]:
#Arrange
arranged_df = df.sort_values(by="Price", ascending=False)
print(arranged_df[["Mileage", "Price"]].head())

      Mileage   Price
3952    13.33  160.00
5620     6.40  120.00
5752    12.50  100.00
1457    12.65   97.07
1917    12.05   93.67


In [184]:
#Summarize
summary_df = df.groupby("Location")["Price"].mean().reset_index()
summary_df = summary_df.rename(columns={"Price": "Average_Price"})
print(summary_df.head())

     Location  Average_Price
0   Ahmedabad       8.567248
1   Bangalore      13.482670
2     Chennai       7.958340
3  Coimbatore      15.160206
4       Delhi       9.881944
